# Take data from CSV and add to items or make new item

In [25]:
import pandas as pd
import re
from datetime import datetime
from wikibaseintegrator import WikibaseIntegrator, wbi_helpers, wbi_login, datatypes
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator.wbi_enums import ActionIfExists
import logging
import json
import random
from pathlib import Path

collections_dir = Path("../wikidata/metadata_collections/")

with Path('../authorization.json').open(mode='r') as authorization_file:
    authorization = json.load(authorization_file)

USER_AUTH = authorization['user']
PASSWORD = authorization['password']
CONSUMER_TOKEN = authorization['consumer_token']
CONSUMER_SECRET = authorization['consumer_secret']

BOTNAME = 'WikidataLiteraryWorksMetaDataUpload [1.0]'
USERNAME = 'Katdav-wd-lit'
    

logging.basicConfig(filename='wikibaseint-debug.log', 
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)


wbi_config['USER_AGENT'] = f'{BOTNAME} (https://www.wikidata.org/wiki/User:{USERNAME})'
wbi_config['MEDIAWIKI_API_URL'] = 'https://www.wikidata.org/w/api.php'

PROPS = {'instance_of':'P31',
        'author':'P50',
         'title':'P1476',
         'has_edition':'P747',
         'edition_of': 'P629',
         'based_on' : 'P144',
         'language':'P407',
         'publication_date':'P577'}
ENTITIES = {
   'literary_work':'Q7725634', 
   'edition' : 'Q3331189',
   'German':'Q188',
   'Hugo_Ball':'Q70989'
}

## Load data from CSV (containing data from Openrefine)

In [5]:
fiction = pd.read_csv(Path(collections_dir, 'de-fiction-wd-1.csv'), index_col=0, dtype={'gutenberg_id':str})
fiction

,author,author_last,author_first,title,gutenberg_id,year,wd,wd_title
0,Arthur Achleitner,Achleitner,Arthur,Der Finanzer,NaN,1916,NaN,Der Finanzer
1,Arthur Achleitner,Achleitner,Arthur,Das Schloß im Moor,NaN,1903,NaN,Das Schloß im Moor
2,Arthur Achleitner,Achleitner,Arthur,Familie Lugmüller,NaN,1896,NaN,Familie Lugmüller
3,Arthur Achleitner,Achleitner,Arthur,Der Bezirkshauptmann. Erster Teil,NaN,1901,NaN,Der Bezirkshauptmann. Erster Teil
4,Arthur Achleitner,Achleitner,Arthur,Geschichten aus den Bergen,NaN,1910,NaN,Geschichten aus den Bergen
...,...,...,...,...,...,...,...,...
4229,Arnold Zweig,Zweig,Arnold,Die Novellen um Claudia,52478,1912,NaN,Die Novellen um Claudia
4230,Friderike Maria Burger Winternitz Zweig,Zweig,Friderike Maria Burger Winternitz,Vögelchen,57114,1919,NaN,Vögelchen
4231,Stefan Zweig,Zweig,Stefan,Amok: Novellen einer Leidenschaft,57850,1922,Q675609,Amok
4232,Stefan Zweig,Zweig,Stefan,Brennendes Geheimnis: Erzählung,24173,1911,NaN,Brennendes Geheimnis: Erzählung


In [6]:
authors = pd.read_csv(Path(collections_dir, 'de_fiction_author_title.csv'), index_col=0, dtype={'gutenberg_id':str})
authors

,author,authorLabel,title
item,,,
http://www.wikidata.org/entity/Q104648471,http://www.wikidata.org/entity/Q19015,Andreas Eschbach,Mutters Blumen
http://www.wikidata.org/entity/Q104648506,http://www.wikidata.org/entity/Q19015,Andreas Eschbach,Eine unberührte Welt
http://www.wikidata.org/entity/Q104650285,http://www.wikidata.org/entity/Q156890,Ernst Barlach,Der Findling
http://www.wikidata.org/entity/Q104698418,http://www.wikidata.org/entity/Q76326,Dietrich Bonhoeffer,Widerstand und Ergebung
http://www.wikidata.org/entity/Q104713924,http://www.wikidata.org/entity/Q84500,Anne Weber,"Annette, ein Heldinnenepos"
...,...,...,...
http://www.wikidata.org/entity/Q136008163,http://www.wikidata.org/entity/Q91458451,Heinrich Brinkmann,Zur Kritik der Widerspiegelungstheorie
http://www.wikidata.org/entity/Q136029569,http://www.wikidata.org/entity/Q15439384,Andrea Böhm,Gott und die Krokodile
http://www.wikidata.org/entity/Q136029575,http://www.wikidata.org/entity/Q1897427,Marie Luise Knott,Verlernen


## Log in to Wikidata

In [32]:
login_instance = wbi_login.Login(user=USER_AUTH, password=PASSWORD)

wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'WikidataLiteraryWorksMetaDataUpload: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'

# Start making new Entities

## Let's do Arthur Kahane, Willkommen und Abschied

In [15]:
fiction[fiction['author'].str.contains("Kahane") & fiction['title'].str.contains("Willkommen")]

,author,author_last,author_first,title,gutenberg_id,year,wd,wd_title
1720,Arthur Kahane,Kahane,Arthur,Willkommen und Abschied,NaN,1919,NaN,Willkommen und Abschied


Unfortunately, our Openrefine results cannot be trusted:

In [50]:

authors[authors['title'] == 'Willkommen und Abschied']

,author,authorLabel,title
item,,,
http://www.wikidata.org/entity/Q1500446,http://www.wikidata.org/entity/Q5879,Johann von Goethe,Willkommen und Abschied


Here is the real QID of Arthur Kahane. 

In [ ]:
kahane_qid = 'Q710106'
kahane = wbi.item.get(entity_id=kahane_qid)
kahane.get_json()

## Make a new work entry

In [33]:
work = fiction[fiction['author'].str.contains("Kahane") & fiction['title'].str.contains("Willkommen")]
fiction[fiction['author'].str.contains("Kahane") & fiction['title'].str.contains("Willkommen")]

,author,author_last,author_first,title,gutenberg_id,year,wd,wd_title
1720,Arthur Kahane,Kahane,Arthur,Willkommen und Abschied,NaN,1919,NaN,Willkommen und Abschied


In [34]:
year = work['year'].item()

def format_year(year : int):
    return datetime(year, 1, 1).strftime("+%Y-%m-%dT%H:%M:%SZ")

formatted_year = format_year(year)
formatted_year

'+1919-01-01T00:00:00Z'

In [35]:
def make_work(author_qid : str, title : str, author_name : str): 
    language = ENTITIES['German']

    new_work = wbi.item.new()
    new_work.labels.set('de', title)
    # Set a default label too
    new_work.labels.set('mul', title)
    new_work.descriptions.set('en', f'A literary work of fiction by {author_name}')
    new_work.descriptions.set('de', f'Ein fiktionales literarisches Werk von {author_name}')

    new_work.claims.add([
        datatypes.Item(value=ENTITIES['literary_work'], prop_nr=PROPS['instance_of']), 
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                    ])
    return new_work

In [ ]:
author_qid = kahane_qid #work[]
title = work['title'].item().strip()
author_name = work['author'].item().strip()

new_work = make_work(author_qid = author_qid, title = title, author_name = author_name)
new_work.get_json()

In [ ]:
new_work = new_work.write()
new_work.id

Let's check what the 'default label' actually looks like so we can add one (It becomes the page title)

In [ ]:
willkommen.labels.get_json()

{'de': {'language': 'de', 'value': 'Willkommen und Abschied'},
 'mul': {'language': 'mul', 'value': 'Willkommen und Abschied'}}

## Make a new edition entry, referencing the work

In [167]:
def make_edition(author_qid : str, title : str, author_name : str, work_qid): 
    language = ENTITIES['German']

    new_edition = wbi.item.new()
    new_edition.labels.set('de', title)
    # Set a default label too
    new_work.labels.set('mul', title)

    new_edition.descriptions.set('en', f'{str(year)} edition of the literary work of fiction by {author_name}')
    new_edition.descriptions.set('de', f'Ausgabe von {str(year)} des fiktionalen literarischen Werks von {author_name}')

    new_edition.claims.add([
        datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                            ])   

    if work_qid: 
        new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                               ])

    return new_edition


In [ ]:
new_edition = make_edition(author_qid = author_qid, 
                        title = title, 
                        author_name = author_name, 
                        work_qid = 'Q136762019')
new_edition.get_json()

In [ ]:
new_edition = new_edition.write()
new_edition.id

In [184]:
def make_gutenberg_edition(author_qid : str, title : str, author_name : str, work_qid, edition_qid):

    language = ENTITIES['German']

    new_edition = wbi.item.new()
    new_edition.labels.set('de', title)
    # Set a default label too
    new_work.labels.set('mul', title)

    new_edition.descriptions.set('en', f'Projekt Gutenberg-DE edition of the literary work of fiction by {author_name}')
    new_edition.descriptions.set('de', f'Ausgabe von Projekt Gutenberg-DE des fiktionalen literarischen Werks von {author_name}')

    new_edition.claims.add([
        datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
        datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of']),
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        # We're not sure about the year, so let's leave it out
        #datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                            ])

    if work_qid: 
        new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                               ])
        
    if edition_qid: 
        new_edition.claims.add([ datatypes.Item(value=edition_qid, prop_nr=PROPS['based_on'])
                               ])   

    return new_edition



In [ ]:
new_gb_edition = make_gutenberg_edition(author_qid = author_qid, 
                           title = title, 
                           author_name = author_name, 
                           work_qid= 'Q136762019',
                           edition_qid = 'Q136762200')
new_gb_edition.get_json()

In [ ]:
new_gb_edition = new_gb_edition.write()
new_gb_edition.id

## Find these new works using SPARQL

In [ ]:
def make_sparql_authors_works(author_qid):
    return '''SELECT ?item ?title ?year (COUNT(?edition) as ?count)
WHERE {
    # is a literary work
  ?item wdt:P31 wd:Q7725634 .
  # author is kahane
  ?item wdt:P50 wd:''' + author_qid + ''' . 
  OPTIONAL {
  ?edition wdt:P629 ?item . }
  # tile in german
  OPTIONAL {
    ?item wdt:P1476 ?title .} 
  OPTIONAL {
    ?item wdt:P577 ?year . }
  }
GROUP BY ?item ?title ?year
ORDER BY DESC(?count)'''


print(make_sparql_authors_works(kahane_qid))

### Kahane now has 1 work with 2 editions on wikidata:

In [ ]:
result = wbi_helpers.execute_sparql_query(make_sparql_authors_works(kahane_qid), 
                                          user_agent=wbi_config['USER_AGENT'])
result